# Read Trouves into pandas

A Trouve is the name of a table. A notebook imports it from its file, in the same way as
any Python module, and clair gives its address in the warehouse. Thus you read the data
of your project by the names that the project uses, and you type no table name.

The work stays divided:

- **The Trouve files and the CLI write.** You edit a Trouve file, and
  `clair run --env dev --select <address>` builds it, tests it, and promotes it.
- **The notebook reads.** It imports the Trouve, and it reads the rows into pandas.

This notebook opens no warehouse connection. An adapter at the end holds its tables in
memory, thus each cell runs on a machine with no Snowflake account. The same calls read
Snowflake when you give them a `SnowflakeAdapter`.

In [1]:
import os
import sys
from pathlib import Path


def find_examples_root(start: Path | None = None) -> Path:
    """Give the directory that holds examples/projects.

    The search starts at the working directory and it goes up. Thus the
    notebook runs from this directory, from the repository root, or from a
    Jupyter server that you started at a different place.
    """
    for directory in [start or Path.cwd(), *(start or Path.cwd()).parents]:
        if (directory / "examples" / "projects").is_dir():
            return directory
    raise FileNotFoundError("No parent directory holds examples/projects.")


PROJECT = find_examples_root() / "examples" / "projects" / "example_4"

# A Trouve file imports its upstream Trouves by their path below the project
# root, thus the root goes on sys.path. Start Jupyter in the project root, and
# Python does this for you.
sys.path.insert(0, str(PROJECT))

# The dev routing entry of example_4 puts each Trouve in the database of the
# person, and it reads the name from CLAIR_USER.
os.environ["CLAIR_USER"] = "alice"

# The CLI configures structlog; a notebook does not. This line keeps the INFO
# messages of each operation out of the cells. Remove it to read them.
import logging

import structlog

structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

## Import the Trouves

Each import is a normal Python import. No discovery runs, and no connection opens.

In [2]:
from example_4_database.derived.daily_event_counts import trouve as daily_event_counts
from example_4_database.derived.top_event_types import trouve as top_event_types
from example_4_database.reference.event_type_labels import trouve as event_type_labels
from example_4_database.source.events import trouve as source_events

str(daily_event_counts.logical_address)

'example_4_database.derived.daily_event_counts'

## The address in each environment

`logical_address` comes from the file path. It is the production address, thus it needs
no environment. `get_physical_address(env_name)` gives the address that
`clair run --env <env_name>` writes to: it reads `__routing__.py` above the file, and it
calls the routing function of a run. The dev entry keeps each SOURCE at its own address,
because clair never builds a SOURCE.

In [3]:
import pandas as pd

trouves = {
    "source.events": source_events,
    "reference.event_type_labels": event_type_labels,
    "derived.daily_event_counts": daily_event_counts,
    "derived.top_event_types": top_event_types,
}

pd.DataFrame(
    [
        {
            "trouve": name,
            "type": trouve.type.value,
            "logical (production)": str(trouve.logical_address),
            "dev": str(trouve.get_physical_address("dev")),
            "prod": str(trouve.get_physical_address("prod")),
        }
        for name, trouve in trouves.items()
    ]
)

,trouve,type,logical (production),dev,prod
0,source.events,source,example_4_database.source.events,example_4_database.source.events,example_4_database.source.events
1,reference.event_type_labels,table,example_4_database.reference.event_type_labels,alice.reference.event_type_labels,example_4_database.reference.event_type_labels
2,derived.daily_event_counts,table,example_4_database.derived.daily_event_counts,alice.derived.daily_event_counts,example_4_database.derived.daily_event_counts
3,derived.top_event_types,table,example_4_database.derived.top_event_types,alice.derived.top_event_types,example_4_database.derived.top_event_types


### The same address as a run

`clair.compile()` routes each Trouve with the same function. Each address of the plan
matches the address of the import. The plan holds each Trouve that a run builds, thus a
SOURCE has no node. `clair.compile()` runs each Trouve file again, thus the cell imports
the Trouve again after it.

In [4]:
import importlib

import clair

plan = clair.compile(PROJECT, env="dev")

for name, trouve in trouves.items():
    if trouve.type.value == "source":
        continue
    module = importlib.import_module(f"example_4_database.{name}")
    planned = plan.node(f"example_4_database.{name}")
    assert planned is not None
    assert module.trouve.get_physical_address("dev") == planned.addresses.physical
    print(f"{name:<30} {planned.addresses.physical}")

reference.event_type_labels    alice.reference.event_type_labels
derived.daily_event_counts     alice.derived.daily_event_counts
derived.top_event_types        alice.derived.top_event_types


## Read the data

`fetch_dataframe()` takes a Trouve, or an address. A Trouve reads its logical address: the
production data. To read the build of an environment, give its physical address. clair
applies no row limit, in the same way as the pandas API of the Snowflake connector.

With Snowflake, the connection is one line. The `dev` profile of
`~/.clair/environments.yml` gives the credentials, and the Trouve still reads production:

```python
from clair import SnowflakeAdapter, load_environment

warehouse = SnowflakeAdapter.connected(load_environment("dev"))
```

This notebook has no account, thus the adapter below holds a DataFrame for each address.
An adapter implements `read_table()`, and `fetch_dataframe()` calls it.

In [5]:
from clair import WarehouseAdapter


class InMemoryWarehouse(WarehouseAdapter):
    """An adapter that holds one DataFrame for each address. It reads, and it writes nothing."""

    def __init__(self, tables: dict[str, pd.DataFrame]) -> None:
        self.tables = tables
        self.read_addresses: list[str] = []

    def read_table(self, address, sample_query=None) -> pd.DataFrame:
        self.read_addresses.append(str(address))
        return self.tables[str(address)].copy()

    # A read needs no connection. The other methods of the contract serve a run,
    # and a notebook that reads calls none of them.
    def connect(self, environment) -> None: ...
    def new_connection(self) -> "InMemoryWarehouse": return self
    def close(self) -> None: ...

    def execute(self, sql): raise NotImplementedError("This notebook only reads.")
    def set_context(self, *arguments, **keyword_arguments): raise NotImplementedError("This notebook only reads.")
    def set_session_parameters(self, *arguments, **keyword_arguments): raise NotImplementedError("This notebook only reads.")
    def describe_object(self, address): raise NotImplementedError("This notebook only reads.")
    def describe_columns(self, address): raise NotImplementedError("This notebook only reads.")
    def describe_query(self, sql): raise NotImplementedError("This notebook only reads.")
    def write_dataframe(self, dataframe, address, transient): raise NotImplementedError("This notebook only reads.")


def daily_counts(rows: list[tuple[str, str, int]]) -> pd.DataFrame:
    dataframe = pd.DataFrame(rows, columns=["event_date", "event_type", "event_count"])
    dataframe["event_date"] = pd.to_datetime(dataframe["event_date"]).dt.date
    return dataframe


warehouse = InMemoryWarehouse(
    {
        # The production data.
        "example_4_database.derived.daily_event_counts": daily_counts(
            [
                ("2024-01-15", "page_view", 120),
                ("2024-01-15", "add_to_cart", 30),
                ("2024-01-15", "purchase", 9),
                ("2024-01-16", "page_view", 140),
                ("2024-01-16", "add_to_cart", 41),
                ("2024-01-16", "purchase", 12),
            ]
        ),
        # The build of `clair run --env dev`, after alice edited the Trouve.
        "alice.derived.daily_event_counts": daily_counts(
            [
                ("2024-01-15", "page_view", 118),
                ("2024-01-15", "add_to_cart", 30),
                ("2024-01-15", "purchase", 9),
                ("2024-01-16", "page_view", 137),
                ("2024-01-16", "add_to_cart", 41),
                ("2024-01-16", "purchase", 12),
            ]
        ),
        "example_4_database.reference.event_type_labels": event_type_labels.dataframe,
    }
)

production_counts = warehouse.fetch_dataframe(daily_event_counts)
production_counts

,event_date,event_type,event_count
0,2024-01-15,page_view,120
1,2024-01-15,add_to_cart,30
2,2024-01-15,purchase,9
3,2024-01-16,page_view,140
4,2024-01-16,add_to_cart,41
5,2024-01-16,purchase,12


### Production against the dev build

Read the dev build at its physical address, and compare the two in pandas. This is the
review of a change: alice edited the Trouve file, `clair run --env dev` built it in her
database, and the notebook shows which rows changed.

In [6]:
dev_counts = warehouse.fetch_dataframe(daily_event_counts.get_physical_address("dev"))

comparison = production_counts.merge(
    dev_counts, on=["event_date", "event_type"], suffixes=("_production", "_dev")
)
comparison["change"] = comparison["event_count_dev"] - comparison["event_count_production"]
comparison[comparison["change"] != 0]

,event_date,event_type,event_count_production,event_count_dev,change
0,2024-01-15,page_view,120,118,-2
3,2024-01-16,page_view,140,137,-3


### Join Trouves in pandas

Each Trouve is a DataFrame now, thus pandas joins them. The seed gives the label of each
event type, and the share of conversions comes out of one `merge`.

In [7]:
labels = warehouse.fetch_dataframe(event_type_labels)

daily_totals = production_counts.merge(labels, on="event_type")
daily_totals.groupby("event_date").apply(
    lambda day: day.loc[day["is_conversion"], "event_count"].sum() / day["event_count"].sum(),
    include_groups=False,
).rename("conversion share").to_frame()

,conversion share
event_date,
2024-01-15,0.056604
2024-01-16,0.062176


In [8]:
# Each read went to the address that clair gave.
warehouse.read_addresses

['example_4_database.derived.daily_event_counts',
 'alice.derived.daily_event_counts',
 'example_4_database.reference.event_type_labels']

## Use your own connection

Put the address in your SQL, and write `.logical_address`. The Trouve itself in an
f-string gives a placeholder token: discovery replaces that token in a Trouve file, and
nothing replaces it in a notebook.

In [9]:
print(f"select * from {daily_event_counts.logical_address}")
print(f"select * from {daily_event_counts}")

select * from example_4_database.derived.daily_event_counts
select * from __CLAIR_TROUVE_4655459408


## Change a Trouve

The notebook does not write. To change `daily_event_counts`, edit its file and build it
in your own database, where the tests run before the promotion:

```bash
clair run --env dev --select example_4_database.derived.daily_event_counts
```

Then read `daily_event_counts.get_physical_address("dev")` again, as the comparison
above does.

The last cell removes the SQL that `clair.compile()` wrote to the artifacts directory of
the example project.

In [10]:
removed = clair.clean(PROJECT).runs
print(f"clair removed {len(removed)} artifact run")

clair removed 1 artifact run
